# Loading Data

To initialize a simulation, we first create a Galaxy object, which accepts data in the following form:
        
        star_cols = {
            "coord"        : (N, 3) numpy array, units: kpc
            "initial_mass" : (N,) numpy array, units: Msun
            "metallicity"  : (N,) numpy array, units: 1
            "age"          : (N,) numpy array, units: Gyr
            "velocity"     : (N, 3) numpy array, units: km/s
                (OPTIONAL -- Includes Doppler shift from velocity relative to the model coordinate system)
        } 
        
        gas_cols = {
            "coord"        : (N, 3) numpy array, units: kpc
            "mass"         : (N,) numpy array, units: Msun
            "metallicity"  : numpy array, units: 1
            "hsml"         : (N,) numpy array, units: kpc
            "temperature"  : (N,) numpy array, units: K
                (OPTIONAL -- Provides the option to ignore gas particles above a maximum temperature value)
            "velocity"     : (N, 3) numpy array, units: km/s
                (OPTIONAL -- Includes Doppler shift from velocity relative to the model coordinate system)
        }  

Data must organized into this form before running SKIRT simulations. Below is a sample script using gizmo_analysis to extract data from a FIRE-2 galaxy:

In [1]:
import numpy as np
import h5py
import os
from astropy.cosmology import Planck13 as cosmo 
import gizmo_analysis as gizmo
import utilities as ut

simulation_directory = "/DFS-L/DATA/cosmo/grenache/pfofie/FIRE2_AGN/M12s/m12f_mass56000_cr_700"
part = gizmo.io.Read.read_snapshots(['star', 'gas'], 'redshift', 0, simulation_directory)


# in utilities.simulation.Snapshot():
* reading:  DFS-L/DATA/cosmo/grenache/pfofie/FIRE2_AGN/M12s/m12f_mass56000_cr_700/snapshot_times.txt

* reading:  DFS-L/DATA/cosmo/grenache/pfofie/FIRE2_AGN/M12s/m12f_mass56000_cr_700/snapshot_scalefactors.txt

* reading:  DFS-L/DATA/cosmo/grenache/pfofie/FIRE2_AGN/M12s/m12f_mass56000_cr_700/snapshot_scale-factors.txt

* input redshift = 0:  using snapshot index = 600, redshift = 0.000


# in gizmo_analysis.gizmo_io.Read():
* reading header from:  DFS-L/DATA/cosmo/grenache/pfofie/FIRE2_AGN/M12s/m12f_mass56000_cr_700/output/snapdir_600/snapshot_600.0.hdf5
  snapshot contains the following number of particles:
    dark      (id = 1): 12037968 particles
    dark2     (id = 2): 3723919 particles
    gas       (id = 0): 10244202 particles
    star      (id = 4): 1835142 particles
    blackhole (id = 5): 0 particles

* reading the following
  species: ['star', 'gas']
  gas properties: ['cosmicray.energy', 'density', 'electron.fraction', 'hydrogen.neutra

In [2]:
star_cols = {}
gas_cols = {}

star_cols["coord"] = part['star'].prop('position')
star_cols["initial_mass"] = part['star'].prop('form.mass')
star_cols["metallicity"] = part['star'].prop('massfraction.metals')
star_cols["age"] = part['star'].prop('age')

gas_cols["coord"] = part['gas'].prop('position')
gas_cols["mass"] = part['gas']['mass']
gas_cols["metallicity"] = part['gas'].prop('massfraction.metals')
gas_cols["hsml"] = part['gas']['size']
gas_cols["temperature"] = part['gas']['temperature']

# get positions relative to host galaxy center
host_index = 0
host_name = ut.catalog.get_host_name(host_index)
star_id = ut.array.get_arange(part['star']['position'].shape[0])
gas_id = ut.array.get_arange(part['gas']['position'].shape[0])

# rotate according to principal axes
star_positions = part['star'].prop(f'{host_name}.distance', star_id)
gas_positions = part['gas'].prop(f'{host_name}.distance', gas_id)

star_cols["coord"] = star_positions
gas_cols["coord"] = gas_positions

import pathlib as Path
import sys
sys.path.append("../src")

import SKIRT_tools
import SKIRT_tools.skirt_tools as sktools

m12f = sktools.Galaxy("m12f_mass56000_cr_700", star_cols, gas_cols)


# in gizmo_analysis.gizmo_star.MassLoss():
* generating 2-D spline to compute stellar mass loss from age & metallicity
  by integrating dM/M / dt over time, *not* accounting for the evolution of M(t)
  number of age bins = 100
  number of metallicity bins = 30
Loading data columns...


We can save this data to an hdf5 file:

In [3]:
m12f.save_to_hdf5("./")

Alternatively, we can load galaxies using previously saved hdf5 files.

In [4]:
m12f_2 = sktools.Galaxy("m12f_mass56000_cr_700.h5", hdf5_file = "m12f_mass56000_cr_700.h5")
print(m12f_2.star_cols)

No snapshot number found.
{'age': array([11.93225159, 10.11775021, 12.39161902, ..., 13.29487812,
       11.53176811, 12.3498231 ], shape=(1835142,)), 'coord': array([[   41.11318146,    97.29458706,   296.29430126],
       [   43.35817803,    88.6173066 ,   293.5521803 ],
       [   55.38968072,    87.48275891,   266.46150306],
       ...,
       [-3289.28467688,  1963.0642463 ,  -342.22793995],
       [-3289.14245072,  1986.1011647 ,  -327.82080986],
       [-3323.99738412,  2016.7489474 ,  -333.46955805]],
      shape=(1835142, 3)), 'initial_mass': array([53540.027, 56370.836, 54769.51 , ..., 63742.12 , 60010.59 ,
       50924.863], shape=(1835142,), dtype=float32), 'metallicity': array([4.5298078e-04, 1.0848382e-03, 2.1563557e-05, ..., 2.0000000e-07,
       1.2366070e-03, 3.5027100e-04], shape=(1835142,), dtype=float32)}
